In [1]:
import numpy as np
import cv2

import os
import glob
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso

In [2]:
def load_images(base_folder):
    print(f"Scanning {base_folder} for images...")
    all_files = glob.glob(os.path.join(base_folder, "*.jpg"))
    if len(all_files) == 0:
        return None, None
        
    data_matrix = np.zeros((len(all_files), 36000))
    file_names = [] # New list to store the real names
    
    row_counter = 0
    for file_path in all_files:
        img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            resized_img = cv2.resize(img, (180, 200))
            data_matrix[row_counter, :] = resized_img.flatten('F')
            
            file_names.append(os.path.basename(file_path))
            row_counter += 1
            
    print(f"SUCCESS: Loaded {row_counter} real face images into the database!")
    return data_matrix[:row_counter, :], np.array(file_names)


def extract_and_flatten_face(image_path, target_width=180, target_height=200):
    raw_img = cv2.imread(image_path)
    if raw_img is None:
        print("ERROR: Could not load image. Please check the file path.")
        return None, None

    gray_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2GRAY)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray_img, scaleFactor=1.1, minNeighbors=5, minSize=(50, 50))
    
    if len(faces) == 0:
        print("Warning: Haar Cascade failed to find a face (likely due to your occlusion).")
        print("Fallback: Using the entire image to test SRC...")
        cropped_face = gray_img
    else:
        x, y, w, h = faces[0]
        cropped_face = gray_img[y:y+h, x:x+w]

    resized_face = cv2.resize(cropped_face, (target_width, target_height))
    face_vector = resized_face.flatten().reshape(-1, 1)
    
    return face_vector, resized_face

def recognize_face_src(test_image_vector, training_matrix, class_labels, tolerance=0.1):
    print("\nRunning L1-minimization optimization...")
    lasso = Lasso(alpha=tolerance, max_iter=10000)
    lasso.fit(training_matrix.T, test_image_vector)
    sparse_x = lasso.coef_
    
    unique_classes = np.unique(class_labels)
    residuals = []
    
    for c in unique_classes:
        class_mask = (class_labels == c)
        x_c = np.zeros_like(sparse_x)
        x_c[class_mask] = sparse_x[class_mask]
        
        reconstructed_image = np.dot(training_matrix.T, x_c)
        error = np.linalg.norm(test_image_vector.flatten() - reconstructed_image)
        residuals.append((c, error))
        
    best_match_class, min_error = min(residuals, key=lambda item: item[1])
    print(f"SRC MATCH FOUND: Class {best_match_class} (Reconstruction Error: {min_error:.2f})")
    return best_match_class



In [3]:

def project_to_eigenspace(img_path, mean_face, Q):
    """Loads an aligned 180x200 image and safely projects it into the eigenface space."""
    # 1. Strip accidental quotes from the Windows path
    clean_path = img_path.strip('\"\'')
    img = cv2.imread(clean_path, cv2.IMREAD_GRAYSCALE)
    
    # 2. Strict validation to prevent NoneType crashes
    if img is None:
        raise ValueError(f"CRITICAL ERROR: Could not load image at {clean_path}")
        
    resized = cv2.resize(img, (180, 200))
    flat_vector = resized.flatten('F').astype(np.float64)
    
    # Center and project: w = Q^T (x - mean)
    centered = flat_vector - mean_face
    weight_vector = np.dot(centered, Q)
    return weight_vector

def calculate_gaussian_kinship(w_child, w_parent, eigenvalues, inheritance_factor=0.5):
    """Evaluates kinship using a vectorized Gaussian Log-Likelihood Ratio test."""
    
    # Add a microscopic epsilon to prevent division by zero on trailing components
    eps = 1e-8
    pop_variance = eigenvalues + eps
    
    # Family variance is a tighter fraction of population variance
    fam_variance = pop_variance * 0.4 
    expected_kin_mean = inheritance_factor * w_parent
    
    # Vectorized Mahalanobis Distances (bypassing np.linalg.inv)
    diff_kin = w_child - expected_kin_mean
    dist_kin = np.sum((diff_kin ** 2) / fam_variance)
    dist_pop = np.sum((w_child ** 2) / pop_variance)
    
    # Vectorized Log-Determinants
    logdet_pop = np.sum(np.log(pop_variance))
    logdet_fam = np.sum(np.log(fam_variance))
    
    llr = 0.5 * (logdet_pop - logdet_fam + dist_pop - dist_kin)
    is_related = llr > 0
    
    print("\n--- Gaussian Likelihood Ratio Test ---")
    print(f"Kinship Mahalanobis Distance : {dist_kin:.2f}")
    print(f"Population Mahalanobis Dist  : {dist_pop:.2f}")
    print(f"Log-Likelihood Ratio (LLR)   : {llr:.2f}")
    print(f"Decision                     : {'RELATED (H1)' if is_related else 'UNRELATED (H0)'}")
    
    return is_related, llr

def check_kinship_both_parents(child_path, parent1_path, parent2_path, mean_face, Q, threshold=4000.0):
    """Evaluates whether the child lies on the convex genetic segment spanned by the parents."""
    wC = project_to_eigenspace(child_path, mean_face, Q)
    wP1 = project_to_eigenspace(parent1_path, mean_face, Q)
    wP2 = project_to_eigenspace(parent2_path, mean_face, Q)
    
    d = wP1 - wP2
    denom = np.dot(d, d)
    
    # Protect against identical parent vectors
    alpha = np.dot(wC - wP2, d) / denom if denom != 0 else 0.5
    
    # Constrain alpha to valid biological range [0, 1]
    alpha_clamped = np.clip(alpha, 0.0, 1.0)
    w_reconstructed = alpha_clamped * wP1 + (1.0 - alpha_clamped) * wP2
    
    residual_error = np.linalg.norm(wC - w_reconstructed)
    is_related = residual_error < threshold
    
    print("\n--- Both-Parent Kinship Test ---")
    print(f"Optimal Genetic Ratio (P1) : {alpha_clamped * 100:.2f}%")
    print(f"Optimal Genetic Ratio (P2) : {(1.0 - alpha_clamped) * 100:.2f}%")
    print(f"Subspace Residual Error    : {residual_error:.2f} (Threshold: {threshold})")
    print(f"Decision                   : {'RELATED' if is_related else 'UNRELATED'}")
    
    return is_related, residual_error, alpha_clamped

def check_kinship_single_parent(child_path, parent_path, mean_face, Q, corr_threshold=0.35):
    """Evaluates kinship using directional eigenspace correlation."""
    wC = project_to_eigenspace(child_path, mean_face, Q)
    wP = project_to_eigenspace(parent_path, mean_face, Q)
    
    norm_C = np.linalg.norm(wC)
    norm_P = np.linalg.norm(wP)
    
    corr = np.dot(wC, wP) / (norm_C * norm_P) if (norm_C != 0 and norm_P != 0) else 0.0
    is_related = corr >= corr_threshold
    
    print("\n--- Single-Parent Kinship Test ---")
    print(f"Eigenspace Correlation (Cosine)  : {corr:.4f} (Threshold: {corr_threshold})")
    print(f"Decision                         : {'RELATED' if is_related else 'UNRELATED'}")
    
    return is_related, corr



In [4]:

X, real_labels = load_images('C:/ISI-D BSDS/SEMESTER-3/PROB-II/Project/Face')

dummy_labels = np.arange(X.shape[0])

error_test_path = r"C:/ISI-D BSDS/SEMESTER-3/PROB-II/Project/Face/WhatsApp Image 2026-09-28 at 3.13.08 PM.jpeg"
vector_data, face_image = extract_and_flatten_face(error_test_path)

#Sparse Representation Classification
if vector_data is not None:
    src_match = recognize_face_src(vector_data, X, real_labels, tolerance=0.1)

    plt.imshow(face_image, cmap='gray')
    plt.title("Extracted Error Face")
    plt.axis('off')
    plt.show()


#Recognize
mean_face = np.mean(X, axis=0)
Y = X - mean_face

surrogate_cov = np.dot(Y, Y.T) / X.shape[0]
eigenvalues, surrogate_eigenvectors = np.linalg.eigh(surrogate_cov)

sort_indices = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sort_indices]
surrogate_eigenvectors = surrogate_eigenvectors[:, sort_indices]

eigenvalues = eigenvalues[:-1]
surrogate_eigenvectors = surrogate_eigenvectors[:, :-1]

#Map the eigenvectors back to the original 36,000-dimensional space
# and normalize them to create the orthonormal basis matrix, Q
P = np.dot(Y.T, surrogate_eigenvectors)

norms = np.linalg.norm(P, axis=0)
Q = P / norms

print(f"Shape of scores matrix: {(scores := np.dot(Y, Q)).shape}")

def recognize_face(test_image_vector, mean_face, Q, database_scores, real_labels, threshold=15000):
    """
    Projects a new face into the Eigenface space and uses Euclidean 
    distance to find the closest match in the database.
    """
    centered_test = test_image_vector - mean_face
    
    test_score = np.dot(centered_test, Q)
    
    distances = np.linalg.norm(database_scores - test_score, axis=1)
    
    min_index = np.argmin(distances)
    min_distance = distances[min_index]
    
    matched_name = real_labels[min_index]
    
    if min_distance < threshold:
        print(f"MATCH FOUND: {matched_name} (Distance: {min_distance:.2f})")
        return matched_name
    else:
        print(f"UNKNOWN: Minimum distance {min_distance:.2f} exceeds threshold.")
        return None



test_file_path = r"C:/Users/nwr-a/OneDrive/Desktop/WhatsApp Image 2026-09-28 at 3.13.09 PM (1).jpeg"

raw_test_img = cv2.imread(test_file_path, cv2.IMREAD_GRAYSCALE)
resized_test_img = cv2.resize(raw_test_img, (180, 200))

real_test_vector = resized_test_img.flatten('F')

match = recognize_face(real_test_vector, mean_face, Q, scores, real_labels, threshold=15000)

def recognize_face_src(test_image_vector, training_matrix, class_labels, tolerance=0.1):
    """
    Implements Sparse Representation-based Classification (SRC) 
    via L1-minimization to bypass occlusions like beards or injuries.
    """
    # Solve the L1-minimization problem: y = Ax + e
    # We use Lasso regression to enforce sparsity on the coefficient vector (x)
    print("Running L1-minimization optimization...")
    lasso = Lasso(alpha=tolerance, max_iter=10000)
    # This aligns with the mathematical model y = Phi * c
    lasso.fit(training_matrix.T, test_image_vector)
    sparse_x = lasso.coef_
    
    unique_classes = np.unique(class_labels)
    residuals = []
    
    for c in unique_classes:
        class_mask = (class_labels == c)
        x_c = np.zeros_like(sparse_x)
        x_c[class_mask] = sparse_x[class_mask]
        
        # FIX 2: Transpose the matrix here as well to reconstruct the 36000-pixel image
        reconstructed_image = np.dot(training_matrix.T, x_c)
        
        error = np.linalg.norm(test_image_vector - reconstructed_image)
        residuals.append((c, error))
        
    best_match_class, min_error = min(residuals, key=lambda item: item[1])
    
    print(f"SRC MATCH FOUND: Class {best_match_class} (Reconstruction Error: {min_error:.2f})")
    return best_match_class



Scanning C:/ISI-D BSDS/SEMESTER-3/PROB-II/Project/Face for images...
SUCCESS: Loaded 52 real face images into the database!
Fallback: Using the entire image to test SRC...

Running L1-minimization optimization...


In [5]:
# 1. Define your specific image inputs here
# Replace these with the actual names of the photos you added to your folder
child_img  = r"C:/ISI-D BSDS/SEMESTER-3/PROB-II/Project/Face/IMG-20250812-WA0016.jpg"
mother_img = r"C:/ISI-D BSDS/SEMESTER-3/PROB-II/Project/Face/WhatsApp Image 2026-09-28 at 3.13.08 PM (1).jpeg"
#father_img = r"C:/ISI-D BSDS/SEMESTER-3/PROB-II/Project/Face/.jpg"

print("\n\n=== STARTING KINSHIP ANALYSIS ===")

# 2. Test Single-Parent Relationship (Gaussian Likelihood)
# We use the eigenvalues (population variance) calculated earlier in the script
try:
    w_child = project_to_eigenspace(child_img, mean_face, Q)
    w_mother = project_to_eigenspace(mother_img, mean_face, Q)
    is_related_single, llr_score = calculate_gaussian_kinship(
        w_child=w_child, 
        w_parent=w_mother, 
        eigenvalues=eigenvalues, 
        inheritance_factor=0.5
    )
except ValueError as e:
    print(e)
'''
# 3. Test Both-Parent Relationship (Convex Subspace Mixture)
try:
    is_related_both, residual, alpha = check_kinship_both_parents(
        child_path=child_img,
        parent1_path=mother_img,
        parent2_path=father_img,
        mean_face=mean_face,
        Q=Q,
        threshold=4000.0  # Threshold will need to be tuned based on your specific camera
    )
except ValueError as e:
    print(e)
'''



=== STARTING KINSHIP ANALYSIS ===

--- Gaussian Likelihood Ratio Test ---
Kinship Mahalanobis Distance : 402.18
Population Mahalanobis Dist  : 51.00
Log-Likelihood Ratio (LLR)   : -152.22
Decision                     : UNRELATED (H0)


'\n# 3. Test Both-Parent Relationship (Convex Subspace Mixture)\ntry:\n    is_related_both, residual, alpha = check_kinship_both_parents(\n        child_path=child_img,\n        parent1_path=mother_img,\n        parent2_path=father_img,\n        mean_face=mean_face,\n        Q=Q,\n        threshold=4000.0  # Threshold will need to be tuned based on your specific camera\n    )\nexcept ValueError as e:\n    print(e)\n'